# Mamdani in Context: Money, Media, and the Coalitions of 2017 and 2021

Part 10 established that Mamdani's November 2025 vote held the outer-borough neighborhoods that had swung to Trump and lost the affluent ones. This notebook puts that result against the two prior mayoral cycles on three dimensions:

1. **Money.** Candidate-committee spending and independent expenditures for and against each major candidate, 2013–2025, from the NYC Campaign Finance Board.
2. **Media.** Daily coverage volume and tone for each major candidate by news source, from the GDELT DOC API, for the 2021 and 2025 cycles (the API begins in 2017; the 2017 race was uncompetitive).
3. **Coalitions.** The same election districts across the 2017 and 2021 generals and Democratic primaries (crosswalked from the 2012 district lines through census blocks), so Mamdani's map can be compared with de Blasio's and Adams's.

**Inputs:** `data/raw/nyc_cfb/*.csv`, `data/raw/gdelt_nyc/gdelt_nyc_daily.csv`, `data/raw/nyc/nyc_ed_history.csv`, `data/raw/nyc/nyc_ed_panel.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables'); FIGURES_DIR = Path('../outputs/figures')
pd.set_option('display.width', 220)

## 1. Money: candidate committees and independent expenditures, 2013–2025

In [2]:
ie = pd.read_csv(RAW_DIR / 'nyc_cfb' / 'independent_expenditures_by_candidate.csv')
fin = pd.read_csv(RAW_DIR / 'nyc_cfb' / 'mayor_candidate_finance_summary.csv')
m = ie[ie.office == 'Mayor'].groupby(['cycle', 'candidate'])[['support', 'oppose']].sum()
m['net_ie'] = m.support - m.oppose
m = m[(m.support + m.oppose) > 250000].sort_values(['cycle', 'support'], ascending=[True, False])
print('Independent expenditures for and against mayoral candidates ($M):'); print((m / 1e6).round(2))
m.to_csv(TABLES_DIR / 'nyc_ie_by_candidate.csv')
fin['cand'] = fin.cand_name.str.replace(r' \(ID.*', '', regex=True)
print('\nCandidate committees ($M): contributions, public funds, spending; contributors; small-dollar share of contributions')
print(fin[['cycle', 'cand', 'net_cntns', 'pubfnd_pmt', 'net_expnd', 'cntrs_no', 'small_share']].to_string(index=False))
# the winner's side vs the best-funded opponent side, each cycle: candidate spending + IE support + IE against the other
def side(cycle, cand_fin, cand_ie):
    f = fin[(fin.cycle == cycle) & fin.cand.str.contains(cand_fin)].net_expnd.sum()
    s = m.loc[(cycle, cand_ie), 'support'] / 1e6 if (cycle, cand_ie) in m.index else 0
    o = m.loc[(cycle, cand_ie), 'oppose'] / 1e6 if (cycle, cand_ie) in m.index else 0
    return f, s, o
rows = []
for cycle, a, ai, b, bi in [(2013, 'de Blasio', 'de Blasio, Bill', 'Quinn', 'Quinn, Christine C'), (2013, 'de Blasio', 'de Blasio, Bill', 'Thompson', 'Thompson, Jr., William C'),
                             (2021, 'Adams, Eric', 'Adams, Eric L', 'Garcia', 'Garcia, Kathryn A'), (2021, 'Adams, Eric', 'Adams, Eric L', 'Yang', 'Yang, Andrew'),
                             (2025, 'Mamdani', 'Mamdani, Zohran K', 'Cuomo', 'Cuomo, Andrew M')]:
    fa, sa, oa = side(cycle, a, ai); fb, sb, ob = side(cycle, b, bi)
    rows.append({'cycle': cycle, 'winner': a, 'winner_committee': fa, 'ie_for_winner': sa, 'ie_against_winner': oa, 'opponent': b, 'opponent_committee': fb, 'ie_for_opponent': sb, 'ie_against_opponent': ob,
                 'winner_side_total': fa + sa + ob, 'opponent_side_total': fb + sb + oa})
sides = pd.DataFrame(rows); sides['opponent_to_winner_ratio'] = sides.opponent_side_total / sides.winner_side_total
print('\nWinner side vs opponent side, $M (committee spending + IE for + IE against the other):'); print(sides.round(2).to_string(index=False))
sides.to_csv(TABLES_DIR / 'nyc_money_sides.csv', index=False)
top = ie[(ie.cycle == 2025) & (ie.office == 'Mayor')].groupby('spender')[['support', 'oppose']].sum()
top['total'] = top.support + top.oppose
print('\n2025 top independent spenders on the mayoral race ($M):'); print((top.sort_values('total', ascending=False).head(12) / 1e6).round(2))

Independent expenditures for and against mayoral candidates ($M):
                                support  oppose  net_ie
cycle candidate                                        
2013  Thompson, Jr., William C     3.08    0.00    3.08
      Quinn, Christine C           0.83    1.00   -0.17
      Lhota, Joseph J              0.52    1.04   -0.52
      de Blasio, Bill              0.51    0.02    0.50
      Liu, John C                  0.45    0.00    0.45
2021  Adams, Eric L                8.04    0.01    8.02
      Donovan, Shaun               6.49    0.00    6.49
      McGuire, Raymond J           5.75    0.00    5.75
      Stringer, Scott M            4.75    0.00    4.75
      Yang, Andrew                 3.85    0.17    3.68
      Wiley, Maya D                1.98    0.00    1.98
2025  Cuomo, Andrew M             31.02    1.67   29.35
      Mamdani, Zohran K            4.83   21.93  -17.10
      Sliwa, Curtis A              0.79    2.82   -2.03
      Adams, Adrienne E            0.5

## 2. Media: coverage volume and tone by source, 2021 vs 2025

GDELT's DOC API returns, per day, the share of all monitored articles matching a query ("volume intensity") and the average tone of those articles (positive minus negative word share, roughly −10 to +10; news is usually negative, so the comparison is relative). Queries are the candidate's name restricted to a source domain; `ALL` is every English-language source GDELT monitors. Tone is averaged over the cycle with daily volume as the weight. Domains GDELT does not index (the pull records them in `empty_series.txt` / `domain_probe.json`) are absent from the tables.

Two coverage gaps in GDELT itself, found in the pull: the index has **no articles from `nypost.com` in 2021** (every 2021 Post query is empty; 2025 is fine), and **no articles at all for June 15 – July 1, 2025** (every 2025 series has 292 of 309 days, the same 17 missing), which is the primary week. The "primary window" below is therefore effectively April 1 – June 14 and the "general window" July 2 – November 4. `nytimes.com` is present in both years but its 2025 volume is a tenth of 2021's (12–18 covered days), and `msnbc.com` is thin in both; treat those columns as unmeasured.


In [3]:
GDELT_OK = (RAW_DIR / 'gdelt_nyc' / 'gdelt_nyc_daily.csv').exists()
if not GDELT_OK:
    print('GDELT file not yet on disk; media section skipped'); raise SystemExit
g = pd.read_csv(RAW_DIR / 'gdelt_nyc' / 'gdelt_nyc_daily.csv')
g['date'] = pd.to_datetime(g.date.astype(str).str[:8])
g = g[g.volume.notna()]
DOMS = [d for d in ['nypost.com', 'nydailynews.com', 'foxnews.com', 'cnn.com', 'msnbc.com', 'nytimes.com', 'ALL'] if d in set(g.domain)]
have = g.groupby(['cycle', 'candidate']).domain.nunique()
print('Series on disk (candidate: domains):'); print(have.to_string())
for f in ['empty_series.txt', 'domain_probe.json']:
    if (RAW_DIR / 'gdelt_nyc' / f).exists(): print(f'\n{f}:', (RAW_DIR / 'gdelt_nyc' / f).read_text().strip())
def wtone(s):
    w = s.volume.fillna(0); t = s.tone
    ok = t.notna() & (w > 0)
    return np.average(t[ok], weights=w[ok]) if ok.any() else np.nan
def summarize(s):
    return pd.Series({'mean_volume': s.volume.mean(), 'days_covered': int((s.volume > 0).sum()), 'tone': wtone(s)})
summ = g.groupby(['cycle', 'candidate', 'domain']).apply(summarize)
tone_tab = summ['tone'].unstack('domain')[DOMS]; vol_tab = summ['mean_volume'].unstack('domain')[DOMS]
print('\nVolume-weighted mean tone by candidate and source, full cycle (Jan 1 to election eve):'); print(tone_tab.round(2))
print('\nMean daily volume intensity (share of all monitored articles, x1000):'); print((vol_tab * 1000).round(3))
tone_tab.to_csv(TABLES_DIR / 'gdelt_nyc_tone.csv'); vol_tab.to_csv(TABLES_DIR / 'gdelt_nyc_volume.csv')
# windows: primaries and generals in each cycle
WIN = {(2021, 'primary'): ('2021-04-01', '2021-06-22'), (2021, 'general'): ('2021-06-23', '2021-11-02'),
       (2025, 'primary'): ('2025-04-01', '2025-06-24'), (2025, 'general'): ('2025-06-25', '2025-11-04')}
wt = {}
for (cyc, lab), (a, b) in WIN.items():
    s = g[(g.cycle == cyc) & (g.date >= a) & (g.date <= b)]
    t = s.groupby(['candidate', 'domain']).apply(wtone).unstack('domain').reindex(columns=DOMS)
    v = s.groupby(['candidate', 'domain']).volume.mean().unstack('domain').reindex(columns=DOMS)
    wt[(cyc, lab)] = t
    print(f'\n{cyc} {lab} window ({a} to {b}): volume-weighted tone'); print(t.round(2))
    print(f'{cyc} {lab} window: mean daily volume x1000'); print((v * 1000).round(3))
    t.to_csv(TABLES_DIR / f'gdelt_nyc_tone_{cyc}_{lab}.csv'); v.to_csv(TABLES_DIR / f'gdelt_nyc_volume_{cyc}_{lab}.csv')


Series on disk (candidate: domains):
cycle  candidate
2021   adams        6
       garcia       6
       sliwa        5
       stringer     6
       wiley        6
       yang         6
2025   adams        7
       cuomo        7
       lander       7
       mamdani      7
       sliwa        7
       stringer     7

empty_series.txt: 2021,adams,nypost.com
2021,garcia,nypost.com
2021,wiley,nypost.com
2021,yang,nypost.com
2021,sliwa,msnbc.com
2021,sliwa,nypost.com
2021,stringer,nypost.com

domain_probe.json: {
 "nypost.com": false,
 "nydailynews.com": false
}

Volume-weighted mean tone by candidate and source, full cycle (Jan 1 to election eve):
domain           nypost.com  nydailynews.com  foxnews.com  cnn.com  msnbc.com  nytimes.com   ALL
cycle candidate                                                                                 
2021  adams             NaN            -1.42        -2.52    -1.46      -1.39        -1.34 -1.86
      garcia            NaN            -1.02        -1.7

In [4]:
if not GDELT_OK: raise SystemExit
# Tone gap: winner minus main opponent, by source, in each window; only pairs with both series on disk
def gap(cycle, a, b, lab):
    t = wt[(cycle, lab)]
    if a not in t.index or b not in t.index: return None
    return (t.loc[a] - t.loc[b]).rename(f'{cycle} {lab}: {a} - {b}')
pairs = [(2021, 'adams', 'garcia', 'primary'), (2021, 'adams', 'wiley', 'primary'), (2021, 'adams', 'yang', 'primary'), (2021, 'adams', 'sliwa', 'general'),
         (2025, 'mamdani', 'cuomo', 'primary'), (2025, 'mamdani', 'lander', 'primary'), (2025, 'mamdani', 'cuomo', 'general'), (2025, 'mamdani', 'sliwa', 'general')]
gaps = pd.concat([x for x in (gap(*p) for p in pairs) if x is not None], axis=1)
print('Tone gap, winner minus opponent, by source (negative = winner covered more negatively):'); print(gaps.round(2))
gaps.to_csv(TABLES_DIR / 'gdelt_nyc_tone_gaps.csv')
# Same candidate, same source, across cycles: the Democratic nominee's coverage in 2021 vs 2025, and the volume ratio
if {'adams', 'mamdani'} <= set(tone_tab.index.get_level_values('candidate')):
    x = pd.DataFrame({'Adams 2021 tone': tone_tab.loc[(2021, 'adams')], 'Mamdani 2025 tone': tone_tab.loc[(2025, 'mamdani')],
                      'Adams 2021 volume x1000': vol_tab.loc[(2021, 'adams')] * 1000, 'Mamdani 2025 volume x1000': vol_tab.loc[(2025, 'mamdani')] * 1000})
    x['volume ratio M/A'] = x['Mamdani 2025 volume x1000'] / x['Adams 2021 volume x1000']
    print('\nDemocratic nominee, 2021 vs 2025, full cycle:'); print(x.round(3))
    x.to_csv(TABLES_DIR / 'gdelt_nyc_nominee_2021_vs_2025.csv')
# Mamdani tone over time, weekly, by source
colors = {'nypost.com': 'tab:red', 'nydailynews.com': 'tab:orange', 'foxnews.com': 'darkred', 'cnn.com': 'tab:blue', 'msnbc.com': 'tab:purple', 'nytimes.com': 'black', 'ALL': 'grey'}
fig, axes = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True)
for dom in DOMS:
    s = g[(g.cycle == 2025) & (g.candidate == 'mamdani') & (g.domain == dom)].set_index('date').sort_index()
    if (s.volume > 0).sum() < 50: continue  # nytimes.com and msnbc.com: a handful of covered days, pure noise weekly
    wk_t = s.resample('W').apply(lambda x: wtone(x) if len(x) else np.nan)
    wk_v = s.volume.resample('W').mean() * 1000
    axes[0].plot(wk_t.index, wk_t.values, label=dom, color=colors[dom], lw=2.5 if dom == 'ALL' else 1.5, alpha=0.9)
    axes[1].plot(wk_v.index, wk_v.values, label=dom, color=colors[dom], lw=2.5 if dom == 'ALL' else 1.5, alpha=0.9)
for ax in axes:
    ax.axvline(pd.Timestamp('2025-06-24'), color='k', ls='--', lw=0.8); ax.grid(alpha=0.3)
    ax.axvspan(pd.Timestamp('2025-06-15'), pd.Timestamp('2025-07-01'), color='0.85', zorder=0)
axes[0].axhline(0, color='k', lw=0.5); axes[0].set_ylabel('GDELT tone (weekly, volume-weighted)'); axes[0].set_title('Coverage of Mamdani by source, 2025 (dashed line: June 24 primary; shaded: no GDELT data June 15 - July 1)'); axes[0].legend(fontsize=8, ncol=4)
axes[1].set_ylabel('volume intensity x1000 (weekly mean)'); axes[1].set_yscale('log')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'gdelt_mamdani_tone_2025.png', dpi=150); print('saved figure')


Tone gap, winner minus opponent, by source (negative = winner covered more negatively):
                 2021 primary: adams - garcia  2021 primary: adams - wiley  2021 primary: adams - yang  2021 general: adams - sliwa  2025 primary: mamdani - cuomo  2025 primary: mamdani - lander  \
domain                                                                                                                                                                                               
nypost.com                                NaN                          NaN                         NaN                          NaN                           1.00                            0.35   
nydailynews.com                         -0.22                        -0.34                        0.19                         0.07                           0.22                           -0.29   
foxnews.com                             -0.39                        -1.00                        0.00                  

## 2b. The Times: a census, and what it says about the GDELT numbers

GDELT crawls; the NYT Archive API returns every article the paper published, so it is a census. That makes it both the right source for the *Times* and a yardstick for GDELT: of the days the paper demonstrably ran a piece on a candidate, on how many did GDELT register any `nytimes.com` volume? It also carries `print_page`, so front-page placement — the one prestige measure available for free, and something GDELT structurally cannot see — is observable.

Articles are matched on candidate name in headline, abstract, lead paragraph or keywords, then restricted to those also containing a race term (`mayor`, `mayoral`, `City Hall`, `primary`, `ranked-choice`). The restriction matters most for Cuomo, whose 2021 coverage is overwhelmingly his governorship and resignation, and for Eric Adams, whose 2025 coverage is largely the indictment.

In [5]:
NYT_F = RAW_DIR / 'nyt_archive' / 'nyt_mayoral_articles.csv'
if not NYT_F.exists():
    print('NYT census not on disk; run src/data_collection/nyt_archive_pull.py (needs NYT_API_KEY)')
else:
    nyt = pd.read_csv(NYT_F); nyt['date'] = pd.to_datetime(nyt.date)
    race = nyt[nyt.race_term]
    def sel(c, cyc): return race[(race.cycle == cyc) & race.candidates.str.contains(c, na=False)]
    rows = []
    for cyc, cands in [(2021, ['adams','yang','wiley','garcia','stringer','sliwa']), (2025, ['mamdani','cuomo','adams','sliwa','lander'])]:
        for c in cands:
            s = sel(c, cyc)
            if not len(s): continue
            rows.append({'cycle': cyc, 'candidate': c, 'articles': len(s), 'A1': int(s.front_page.sum()),
                         'A1_share': s.front_page.mean(), 'median_words': s.word_count.median()})
    tab = pd.DataFrame(rows)
    print('NYT race-related coverage by candidate (census, not a crawl sample):'); print(tab.to_string(index=False))
    tab.to_csv(TABLES_DIR / 'nyt_coverage_by_candidate.csv', index=False)
    # nominee comparison, split at each cycle's primary
    PRIM = {2021: pd.Timestamp('2021-06-22'), 2025: pd.Timestamp('2025-06-24')}
    split = []
    for cyc, c in [(2021, 'adams'), (2025, 'mamdani')]:
        s = sel(c, cyc); pre, post = s[s.date <= PRIM[cyc]], s[s.date > PRIM[cyc]]
        split.append({'cycle': cyc, 'nominee': c, 'pre_articles': len(pre), 'pre_A1': int(pre.front_page.sum()),
                      'post_articles': len(post), 'post_A1': int(post.front_page.sum()),
                      'total_articles': len(s), 'total_A1': int(s.front_page.sum())})
    sp = pd.DataFrame(split)
    print('\nDemocratic nominee, before and after the primary:'); print(sp.to_string(index=False))
    sp.to_csv(TABLES_DIR / 'nyt_nominee_before_after_primary.csv', index=False)
    # GDELT recall for nytimes.com, benchmarked against the census
    rec = []
    for cyc, c in [(2021, 'adams'), (2025, 'mamdani'), (2025, 'cuomo'), (2021, 'yang')]:
        s = sel(c, cyc)
        gd = g[(g.cycle == cyc) & (g.candidate == c) & (g.domain == 'nytimes.com') & (g.volume > 0)]
        nd, gdd = set(s.date.dt.date), set(gd.date.dt.date)
        rec.append({'cycle': cyc, 'candidate': c, 'nyt_publication_days': len(nd), 'gdelt_days': len(gdd),
                    'overlap': len(nd & gdd), 'gdelt_recall': len(nd & gdd) / max(len(nd), 1)})
    rc = pd.DataFrame(rec)
    print('\nGDELT recall for nytimes.com, against the census:'); print(rc.round(3).to_string(index=False))
    rc.to_csv(TABLES_DIR / 'gdelt_nyt_recall.csv', index=False)
    print('\n-> GDELT holds 2021 nytimes.com well and 2025 barely at all; the 2025 Times column in the')
    print('   GDELT tables above is a crawl artifact and is not interpreted anywhere in the report.')
    # A1 timeline
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for cyc, c, col in [(2021, 'adams', 'tab:blue'), (2025, 'mamdani', 'tab:green')]:
        s = sel(c, cyc); a1 = s[s.front_page]
        day = (a1.date - PRIM[cyc]).dt.days
        ax.scatter(day, [f'{c.title()} {cyc}'] * len(day), s=60, alpha=0.75, color=col, edgecolors='none')
    ax.axvline(0, color='k', ls='--', lw=1); ax.text(2, -0.42, 'primary', fontsize=8)
    ax.set_xlabel('days from the primary'); ax.set_title('New York Times front-page (A1) stories on the Democratic nominee')
    ax.grid(alpha=0.3, axis='x'); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'nyt_a1_timeline.png', dpi=150)
    print('saved figure')

NYT race-related coverage by candidate (census, not a crawl sample):
 cycle candidate  articles  A1  A1_share  median_words
  2021     adams       141  25  0.177305        1100.0
  2021      yang        69   7  0.101449        1066.0
  2021     wiley        36   3  0.083333        1008.5
  2021    garcia        29   2  0.068966        1049.0
  2021  stringer        19   2  0.105263        1177.0
  2021     sliwa        41   2  0.048780         910.0
  2025   mamdani       582  36  0.061856        1094.5
  2025     cuomo       391  21  0.053708        1132.0
  2025     adams       246  17  0.069106         928.5
  2025     sliwa        94   3  0.031915        1041.0
  2025    lander        17   0  0.000000        1217.0

Democratic nominee, before and after the primary:
 cycle nominee  pre_articles  pre_A1  post_articles  post_A1  total_articles  total_A1
  2021   adams            51      12             90       13             141        25
  2025 mamdani            81       3          

## 2c. Tone on the complete population, and what it can and cannot be used for

GDELT is a crawl sample; the NYT archive is a census. So tone can be computed on the population directly, with no sampling error, and every within-*Times* comparison below is exact rather than estimated.

Two things constrain how far that goes. The archive returns headline and abstract — a median of 31 words — not article body, so this scores a different and much shorter text than GDELT's full-article tone. (`lead_paragraph` is present for 98.7% of 2021 records and 0% of 2025 ones, so it is excluded: including it would score the cycles on different amounts of text.) And the scorer is VADER, not GDELT's lexicon.

The first cell below therefore validates before it concludes: on 2021 `nytimes.com`, where GDELT recall is 82–92%, how well does census tone track GDELT tone on the same outlet-days? The answer sets the ceiling on what this measure may be used for.

In [6]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy import stats
_an = SentimentIntensityAnalyzer()
nyt2 = pd.read_csv(RAW_DIR / 'nyt_archive' / 'nyt_mayoral_articles.csv'); nyt2['date'] = pd.to_datetime(nyt2.date)
nyt2['tone_c'] = nyt2.text.fillna('').map(lambda t: _an.polarity_scores(t)['compound'])
rc2 = nyt2[nyt2.race_term]
def sel2(c, cyc): return rc2[(rc2.cycle == cyc) & rc2.candidates.str.contains(c, na=False)]

# --- validation: census tone vs GDELT tone, 2021 nytimes.com, at three aggregations
val = []
for freq, lab in [('D', 'daily'), ('W', 'weekly'), ('M', 'monthly')]:
    A, B = [], []
    for c in ['adams', 'yang', 'wiley', 'garcia', 'stringer', 'sliwa']:
        s = sel2(c, 2021)
        gs = g[(g.cycle == 2021) & (g.candidate == c) & (g.domain == 'nytimes.com') & (g.volume > 0)]
        if len(s) < 5 or len(gs) < 5: continue
        a = s.set_index('date').tone_c.resample(freq).mean(); b = gs.set_index('date').tone.resample(freq).mean()
        j = pd.concat([a.rename('census'), b.rename('gdelt')], axis=1).dropna()
        A.append(j.census); B.append(j.gdelt)
    x, y = pd.concat(A), pd.concat(B)
    r, p = stats.pearsonr(x, y)
    val.append({'aggregation': lab, 'n': len(x), 'pearson_r': r, 'R2': r**2, 'p': p})
v = pd.DataFrame(val)
print('VALIDATION -- census tone (headline+abstract, VADER) vs GDELT tone (full text), 2021 nytimes.com:')
print(v.round(4).to_string(index=False))
b_, a_, rv, pv, se_ = stats.linregress(x, y)
print(f'\nmonthly regression GDELT = {a_:+.2f} + {b_:+.2f}*census; residual SD {np.std(y-(a_+b_*x)):.2f} GDELT points')
print('-> the two agree on trend and barely at all day to day. Mapping census tone onto the GDELT')
print('   scale would carry a residual larger than every tone difference in Part 10, so it is not done:')
print('   census tone is reported as its own measure and compared only within itself.')
v.to_csv(TABLES_DIR / 'nyt_census_tone_validation.csv', index=False)

# --- is GDELT's missingness systematic? (only testable where recall is high)
cov21 = set(g[(g.cycle==2021)&(g.candidate=='adams')&(g.domain=='nytimes.com')&(g.volume>0)].date.dt.date)
a21 = sel2('adams', 2021).copy(); a21['caught'] = a21.date.dt.date.isin(cov21)
tt, pp = stats.ttest_ind(a21[a21.caught].tone_c, a21[~a21.caught].tone_c, equal_var=False)
print(f"\nMISSINGNESS -- Adams 2021 articles on days GDELT saw ({a21.caught.sum()}) vs missed ({(~a21.caught).sum()}):")
print(f"  census tone {a21[a21.caught].tone_c.mean():+.3f} vs {a21[~a21.caught].tone_c.mean():+.3f} (p={pp:.3f}); "
      f"A1 share {(a21[a21.caught].print_page==1).mean():.1%} vs {(a21[~a21.caught].print_page==1).mean():.1%}")
print('  no evidence of systematic loss, but 16 missed articles is far too few to establish randomness,')
print('  and this cannot be tested for 2025 (94% missing) or for any outlet without a census.')

# --- substantive comparisons, complete population
PRIM = {2021: pd.Timestamp('2021-06-22'), 2025: pd.Timestamp('2025-06-24')}
rows = []
for cyc, cands in [(2021, ['adams','yang','wiley','garcia','sliwa']), (2025, ['mamdani','cuomo','adams','sliwa'])]:
    for c in cands:
        s = sel2(c, cyc)
        if len(s) < 15: continue
        rows.append({'cycle': cyc, 'candidate': c, 'n': len(s), 'census_tone': s.tone_c.mean(),
                     'se': s.tone_c.std()/np.sqrt(len(s)), 'share_negative': (s.tone_c < 0).mean()})
ct = pd.DataFrame(rows)
print('\nCENSUS TONE BY CANDIDATE (complete population of race-related Times articles):')
print(ct.round(3).to_string(index=False))
ct.to_csv(TABLES_DIR / 'nyt_census_tone.csv', index=False)
cmp_ = []
for lab, (c1, y1), (c2, y2) in [('nominee across cycles', ('adams',2021), ('mamdani',2025)),
                                ('2025 pre-primary', ('mamdani',2025), ('cuomo',2025)),
                                ('2025 post-primary', ('mamdani',2025), ('cuomo',2025))]:
    s1, s2 = sel2(c1, y1), sel2(c2, y2)
    if 'pre-primary' in lab:  s1, s2 = s1[s1.date <= PRIM[2025]], s2[s2.date <= PRIM[2025]]
    if 'post-primary' in lab: s1, s2 = s1[s1.date > PRIM[2025]], s2[s2.date > PRIM[2025]]
    t_, p_ = stats.ttest_ind(s1.tone_c, s2.tone_c, equal_var=False)
    cmp_.append({'comparison': lab, 'a': f'{c1} {y1}', 'a_tone': s1.tone_c.mean(), 'a_n': len(s1),
                 'b': f'{c2} {y2}', 'b_tone': s2.tone_c.mean(), 'b_n': len(s2),
                 'gap': s1.tone_c.mean()-s2.tone_c.mean(), 'p': p_})
cp = pd.DataFrame(cmp_)
print('\nCOMPARISONS:'); print(cp.round(3).to_string(index=False))
cp.to_csv(TABLES_DIR / 'nyt_census_tone_comparisons.csv', index=False)
m25 = sel2('mamdani', 2025)
pre, post = m25[m25.date <= PRIM[2025]], m25[m25.date > PRIM[2025]]
t_, p_ = stats.ttest_ind(pre.tone_c, post.tone_c, equal_var=False)
print(f"\nMamdani's own Times tone: pre-primary {pre.tone_c.mean():+.3f} (n={len(pre)}), "
      f"post {post.tone_c.mean():+.3f} (n={len(post)}), p={p_:.3f}")

# --- what is actually being scored, and does it survive attack ---
mam = sel2('mamdani', 2025).copy(); mam['w'] = mam.text.fillna('').str.split().str.len()
print(f"\nSCORED TEXT: headline + abstract, median {mam.w.median():.0f} words against a median "
      f"article of {mam.word_count.median():.0f} -> {mam.w.median()/mam.word_count.median():.1%} of the article.")
print(f"  {(mam.tone_c == 0).mean():.1%} of Mamdani articles contain no lexicon word at all and score exactly 0.")
_lex = dict(_an.lexicon)
# (a) political vocabulary VADER reads as sentiment: 'party' (+1.7) means a celebration to VADER
AMBIG = ['party','top','fight','star','hunger','share','critical','crown','rally','union']
# (b) horse-race vocabulary: describes the result, not treatment of the candidate
OUTCOME = ['won','win','wins','winner','winning','victory','defeat','defeated','lost','lose','loss',
           'leading','lead','ahead','surge','surged','triumph','beat','upset','endorsement','endorsed','support','supporters']
for name, drop in [('no political words', AMBIG), ('no outcome words', OUTCOME)]:
    for w in drop: _an.lexicon.pop(w, None)
    rc2[f'tone_{name}'] = rc2.text.fillna('').map(lambda t: _an.polarity_scores(t)['compound'])
    _an.lexicon.clear(); _an.lexicon.update(_lex)
rob = []
for name in ['tone_c', 'tone_no political words', 'tone_no outcome words']:
    a, b = sel2('adams',2021), sel2('mamdani',2025)
    m2 = sel2('mamdani',2025); pre, post = m2[m2.date <= PRIM[2025]], m2[m2.date > PRIM[2025]]
    rob.append({'scoring': name.replace('tone_',''), 'Mamdani25_minus_Adams21': b[name].mean()-a[name].mean(),
                'Mamdani_post_minus_pre': post[name].mean()-pre[name].mean(),
                'p_post_pre': stats.ttest_ind(pre[name], post[name], equal_var=False)[1]})
rb = pd.DataFrame(rob)
print('\nROBUSTNESS -- do the comparisons depend on words VADER misreads in a political context?')
print(rb.round(3).to_string(index=False))
print('  Levels are crude; the differences are not sensitive to either attack, because the misread')
print('  words appear at the same rate across candidates and windows and cancel in a comparison.')
rb.to_csv(TABLES_DIR / 'nyt_census_tone_robustness.csv', index=False)

VALIDATION -- census tone (headline+abstract, VADER) vs GDELT tone (full text), 2021 nytimes.com:
aggregation   n  pearson_r     R2      p
      daily 181     0.3327 0.1107 0.0000
     weekly  97     0.4302 0.1851 0.0000
    monthly  40     0.5570 0.3102 0.0002

monthly regression GDELT = -1.37 + +2.12*census; residual SD 1.06 GDELT points
-> the two agree on trend and barely at all day to day. Mapping census tone onto the GDELT
   scale would carry a residual larger than every tone difference in Part 10, so it is not done:
   census tone is reported as its own measure and compared only within itself.

MISSINGNESS -- Adams 2021 articles on days GDELT saw (126) vs missed (15):
  census tone +0.092 vs -0.024 (p=0.455); A1 share 19.0% vs 13.3%
  no evidence of systematic loss, but 16 missed articles is far too few to establish randomness,
  and this cannot be tested for 2025 (94% missing) or for any outlet without a census.

CENSUS TONE BY CANDIDATE (complete population of race-related Ti

## 2d. Salience instead of valence: what the coverage raised

Sentiment scoring is the wrong instrument for the question people actually ask about coverage. It reads isolated words: the *Times* piece headlined "Mamdani's Delicate Quest to Become New York City's First Muslim Mayor" scores −0.59, but on *war* and *criticism*, not on the choice to lead with his religion or on the description of his Gaza position as drawing "criticism from across the political spectrum." A measure that lands on the right sign for irrelevant reasons is worse than one that is plainly wrong, because it looks like confirmation.

What *is* measurable on this text is **salience** — what a piece raises. And the thin text works in favour here rather than against: headline and abstract is precisely where framing sits, so scoring 3.2% of the article is far less damaging for "what is this about" than for "is this positive." A term appearing in a headline is unambiguous in a way valence never is.

So: for each candidate, the share of their articles whose headline or abstract raises religion, the Israel–Gaza conflict, or socialism — against every other candidate in both cycles as the control.

In [7]:
FRAMES = {
 'religion': r'\bmuslim|islam|mosque|jewish|synagogue|catholic|christian',
 'Israel/Gaza': r'\bisrael|gaza|palestin|zionis|intifada|antisemit|anti-semit|hamas',
 'socialism/radical': r'\bsocialis|marxis|communis|radical|far[- ]left|extremis',
 'crime/safety': r'\bcrime|police|nypd|safety|violent|shooting',
 'affordability': r'\brent|grocer|afford|childcare|child care|minimum wage|housing',
 'corruption': r'\bcorrupt|scandal|indict|charges|bribe|fraud|harassment|misconduct',
}
sal = rc2.copy(); sal['t'] = sal.text.fillna('').str.lower()
for f, pat in FRAMES.items():
    sal[f] = sal.t.str.contains(pat, regex=True)
rows = []
for c, cyc in [('mamdani',2025),('cuomo',2025),('adams',2025),('sliwa',2025),('lander',2025),
               ('adams',2021),('yang',2021),('wiley',2021),('garcia',2021),('sliwa',2021)]:
    s = sal[(sal.cycle == cyc) & sal.candidates.str.contains(c, na=False)]
    if len(s) < 15: continue
    rows.append({'cycle': cyc, 'candidate': c, 'n': len(s), **{f: s[f].mean() for f in FRAMES}})
S = pd.DataFrame(rows)
print('SHARE OF A CANDIDATE\'S TIMES ARTICLES RAISING EACH FRAME (headline + abstract):')
print(S.assign(**{f: (S[f]*100).round(1) for f in FRAMES}).to_string(index=False))
S.to_csv(TABLES_DIR / 'nyt_frame_salience.csv', index=False)
# Mamdani against the rest of his own field
m_, o_ = sal[(sal.cycle==2025) & sal.candidates.str.contains('mamdani')], sal[(sal.cycle==2025) & ~sal.candidates.str.contains('mamdani')]
print('\nMAMDANI vs THE REST OF THE 2025 FIELD:')
cmp2 = []
for f in FRAMES:
    t_, p_ = stats.ttest_ind(m_[f].astype(float), o_[f].astype(float), equal_var=False)
    cmp2.append({'frame': f, 'mamdani': m_[f].mean(), 'others_2025': o_[f].mean(), 'diff': m_[f].mean()-o_[f].mean(), 'p': p_})
    print(f"  {f:20s} {m_[f].mean():6.1%}  vs {o_[f].mean():6.1%}   diff {m_[f].mean()-o_[f].mean():+6.1%}  p={p_:.4f}")
pd.DataFrame(cmp2).to_csv(TABLES_DIR / 'nyt_frame_salience_tests.csv', index=False)
# the novelty confound, quantified
REL = r'\bmuslim|islam|mosque'
mrel = m_[m_.t.str.contains(REL, regex=True)]
firsts = mrel[mrel.t.str.contains(r'\bfirst muslim|first south asian|youngest', regex=True)]
print(f"\nNOVELTY CONFOUND: Mamdani is the first Muslim nominee, so some identity coverage is simply news.")
print(f"  articles raising Muslim/Islam: {len(mrel)} ({len(mrel)/len(m_):.1%} of his coverage)")
print(f"  of those, using a 'first Muslim / first South Asian / youngest' hook: {len(firsts)}")
print(f"  raising it WITHOUT that hook: {len(mrel)-len(firsts)}")
print(f"  religion raised for every other candidate, both cycles: "
      f"{', '.join(f'{r.candidate} {r.cycle} {r.religion:.1%}' for r in S.itertuples() if not (r.candidate=='mamdani'))}")

SHARE OF A CANDIDATE'S TIMES ARTICLES RAISING EACH FRAME (headline + abstract):
 cycle candidate   n  religion  Israel/Gaza  socialism/radical  crime/safety  affordability  corruption
  2025   mamdani 582       7.0          5.7                4.3           3.4            6.9         1.2
  2025     cuomo 391       3.8          3.8                2.0           2.6            6.1         3.1
  2025     adams 246       0.8          1.6                0.0           7.7            5.7        29.3
  2025     sliwa  94       1.1          1.1                1.1           4.3            4.3         2.1
  2025    lander  17       0.0          0.0                0.0           5.9            5.9         0.0
  2021     adams 141       0.0          0.0                0.0          16.3            3.5         0.0
  2021      yang  69       1.4          1.4                1.4           8.7            0.0         1.4
  2021     wiley  36       0.0          0.0                0.0          11.1            

## 3. Coalitions: the same districts in 2017, 2021, and 2025

2017 and 2021 results are on the 2012 district lines and are crosswalked to current districts through 2020 census blocks (population-weighted). The comparison is of each winner's share against the demographic type of the district and against the 2020→2024 presidential swing.

In [8]:
d = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_panel.csv', index_col='ElectDist')
h = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_history.csv', index_col='ElectDist')
d = d.join(h, how='left')
d['pres24_2p'] = d.pres24_harris + d.pres24_trump; d['pres20_2p'] = d.pres20_biden + d.pres20_trump; d['gen25_2p'] = d.gen25_mamdani + d.gen25_cuomo
d['trump_share'] = d.pres24_trump / d.pres24_2p; d['swing_to_trump'] = d.trump_share - d.pres20_trump / d.pres20_2p
d['mamdani_gen_share'] = d.gen25_mamdani / d.gen25_total
d['mamdani_h2h_prim'] = d.prim25_h2h_mamdani / (d.prim25_h2h_mamdani + d.prim25_h2h_cuomo)
d['deblasio17_share'] = d.gen17_deblasio / d.gen17_total
d['deblasio17_prim_share'] = d.prim17_deblasio / d.prim17_total
d['adams21_share'] = d.gen21_adams / d.gen21_total
d['adams_h2h_garcia'] = d.prim21_h2h_adams_v_garcia / (d.prim21_h2h_adams_v_garcia + d.prim21_h2h_garcia_v_adams)
d['adams_h2h_wiley'] = d.prim21_h2h_adams_v_wiley / (d.prim21_h2h_adams_v_wiley + d.prim21_h2h_wiley_v_adams)
d['wiley_first_share'] = d.prim21_first_wiley / d.prim21_ballots
d['adams_first_share'] = d.prim21_first_adams / d.prim21_ballots
ok = (d.pres24_2p >= 50) & (d.gen25_2p >= 50) & (d.pres20_2p >= 50) & (d.gen21_total >= 50) & (d.gen17_total >= 50) & (~d.combined_into.fillna(False)) & d.pop2020.gt(0)
e = d[ok].copy(); print('EDs usable across 2017, 2020, 2021, 2024, 2025:', len(e))
e['majority'] = np.where(e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].max(axis=1) >= 0.5,
                         e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].idxmax(axis=1).str.replace('_share', '').str.replace('_nh', ''), 'mixed')
e['swing_decile'] = pd.qcut(e.swing_to_trump, 10, labels=False) + 1
e['income_tercile'] = pd.qcut(e.med_income, 3, labels=['low', 'mid', 'high'])
def agg(s):
    return pd.Series({'N': len(s), 'deBlasio_gen17': s.gen17_deblasio.sum()/s.gen17_total.sum(), 'deBlasio_prim17': s.prim17_deblasio.sum()/s.prim17_total.sum(),
        'Adams_gen21': s.gen21_adams.sum()/s.gen21_total.sum(), 'Adams_v_Garcia_prim21': s.prim21_h2h_adams_v_garcia.sum()/(s.prim21_h2h_adams_v_garcia.sum()+s.prim21_h2h_garcia_v_adams.sum()),
        'Adams_v_Wiley_prim21': s.prim21_h2h_adams_v_wiley.sum()/(s.prim21_h2h_adams_v_wiley.sum()+s.prim21_h2h_wiley_v_adams.sum()), 'Wiley_first21': s.prim21_first_wiley.sum()/s.prim21_ballots.sum(),
        'Harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'Mamdani_gen25': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'Mamdani_v_Cuomo_prim25': s.prim25_h2h_mamdani.sum()/(s.prim25_h2h_mamdani.sum()+s.prim25_h2h_cuomo.sum()),
        'votes_gen17': s.gen17_total.sum(), 'votes_gen21': s.gen21_total.sum(), 'votes_gen25': s.gen25_total.sum(), 'votes_prim21': s.prim21_ballots.sum(), 'votes_prim25': s.prim25_ballots.sum()})
by_maj = e.groupby('majority').apply(agg); print('By majority group:'); print(by_maj.round(3).T)
by_dec = e.groupby('swing_decile').apply(agg); print('\nBy 2020->2024 swing decile:'); print(by_dec.round(3).T)
by_inc = e.groupby('income_tercile').apply(agg); print('\nBy income tercile:'); print(by_inc.round(3).T)
by_maj.to_csv(TABLES_DIR / 'nyc_history_by_majority.csv'); by_dec.to_csv(TABLES_DIR / 'nyc_history_by_swing_decile.csv'); by_inc.to_csv(TABLES_DIR / 'nyc_history_by_income.csv')

EDs usable across 2017, 2020, 2021, 2024, 2025: 3790
By majority group:
majority                     asian       black    hispanic       mixed       white
N                          215.000     624.000     705.000    1029.000    1217.000
deBlasio_gen17               0.542       0.920       0.831       0.685       0.501
deBlasio_prim17              0.643       0.897       0.835       0.739       0.605
Adams_gen21                  0.516       0.909       0.758       0.669       0.578
Adams_v_Garcia_prim21        0.457       0.841       0.624       0.500       0.294
Adams_v_Wiley_prim21         0.566       0.713       0.599       0.507       0.456
Wiley_first21                0.126       0.200       0.204       0.247       0.215
Harris_24                    0.530       0.899       0.702       0.693       0.617
Mamdani_gen25                0.465       0.636       0.584       0.578       0.414
Mamdani_v_Cuomo_prim25       0.614       0.438       0.553       0.646       0.562
votes_gen17    

In [9]:
# Which past coalition does Mamdani's resemble? ED-level correlations of winner shares, weighted by 2025 votes
def wcorr(x, y, w):
    m = x.notna() & y.notna() & w.notna(); x, y, w = x[m], y[m], w[m]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.average((x-mx)*(y-my), weights=w) / np.sqrt(np.average((x-mx)**2, weights=w) * np.average((y-my)**2, weights=w))
cols = ['deblasio17_share', 'deblasio17_prim_share', 'adams21_share', 'adams_h2h_garcia', 'adams_h2h_wiley', 'wiley_first_share', 'harris_share', 'mamdani_gen_share', 'mamdani_h2h_prim']
e['harris_share'] = e.pres24_harris / e.pres24_2p
C = pd.DataFrame({a: {b: wcorr(e[a], e[b], e.gen25_total) for b in cols} for a in cols})
print('ED-level correlation matrix (weighted by 2025 votes):'); print(C.round(2))
C.to_csv(TABLES_DIR / 'nyc_history_correlations.csv')
# Turnout across cycles by group
print('\nGeneral-election votes by majority group, 2017 / 2021 / 2025 (thousands) and 2025/2021 ratio:')
t = by_maj[['votes_gen17', 'votes_gen21', 'votes_gen25']] / 1000; t['ratio_25_21'] = by_maj.votes_gen25 / by_maj.votes_gen21; print(t.round(2))
print('\nPrimary ballots by majority group, 2021 / 2025 (thousands):'); print((by_maj[['votes_prim21', 'votes_prim25']] / 1000).round(1).assign(ratio=(by_maj.votes_prim25 / by_maj.votes_prim21).round(2)))

ED-level correlation matrix (weighted by 2025 votes):
                       deblasio17_share  deblasio17_prim_share  adams21_share  adams_h2h_garcia  adams_h2h_wiley  wiley_first_share  harris_share  mamdani_gen_share  mamdani_h2h_prim
deblasio17_share                   1.00                   0.94           0.91              0.30            -0.19               0.48          0.79               0.80              0.26
deblasio17_prim_share              0.94                   1.00           0.82              0.43            -0.03               0.34          0.69               0.70              0.16
adams21_share                      0.91                   0.82           1.00              0.13            -0.24               0.42          0.87               0.72              0.19
adams_h2h_garcia                   0.30                   0.43           0.13              1.00             0.81              -0.39         -0.08              -0.09             -0.54
adams_h2h_wiley                

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, (col, lab) in zip(axes, [('deblasio17_share', 'de Blasio, Nov 2017'), ('adams21_share', 'Adams, Nov 2021'), ('mamdani_gen_share', 'Mamdani, Nov 2025')]):
    for grp, c in [('hispanic', 'tab:orange'), ('black', 'tab:purple'), ('white', 'tab:blue'), ('asian', 'tab:green'), ('mixed', 'grey')]:
        s = e[e.majority == grp]
        ax.scatter(s.swing_to_trump, s[col], s=np.sqrt(s.pres24_2p)/2, alpha=0.3, c=c, label=grp, edgecolors='none')
    ax.set_title(lab); ax.set_xlabel('Swing to Trump 2020->2024'); ax.set_xlim(-0.15, 0.45); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.axvline(0, color='k', lw=0.5)
axes[0].set_ylabel('Winner share of mayoral vote'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'nyc_three_mayors_by_swing.png', dpi=150); print('saved figure')

saved figure


## Limitations

- **GDELT tone is a crude lexicon score** on article text; it does not distinguish an article attacking a candidate from one reporting attacks on them, and it is not a measure of what voters saw. Domain restriction captures each outlet's website, not its broadcast. 2021 is the only comparable cycle.
- **IE totals are as reported to the CFB** through the current filing; spending reported under "not determined" or on ballot proposals is excluded. Federal-style dark money outside CFB jurisdiction is not captured.
- **2017 and 2021 results are crosswalked** from the 2012 district lines; the ≥ 50-vote filter in every contest drops ~10% of districts, disproportionately low-turnout ones.
- **2013 is not on disk** (PDF only).